# Evaluación Científica Definitiva — `SM-CFHR` / `ASHR` (`AdaptiveStructuredHuberRegressor`)

Este cuaderno constituye el documento computacional central de la evaluación científica de **SM-CFHR** (*Structured Monotone Capacity-Factor Huber Regressor*). Reproduce íntegramente el análisis, tablas, figuras y auditorías a partir de los artefactos OOF definitivos cerrados en `resultados/evaluacion_ashr_oof/` **sin volver a entrenar** los 28 escenarios:
- **Estado A (`PENDING`):** sin resultados definitivos en `resultados/evaluacion_ashr_oof/`; presenta formulación, protocolo, compatibilidad `scikit-learn` e inventario sin mostrar cifras ficticias.
- **Estado B (`VALID`):** con manifiesto válido y 28 escenarios completos (`Fixed` + `HPO`), verifica integridad SHA-256, audita alineación, Optuna y Nested CV, ejecuta inferencia estadística y consolida tablas/figuras.
- **Estado C (`INVALID` / parcial):** detecta ausencias o inconsistencias de hash/esquema y bloquea conclusiones confirmatorias.

## 1. Introducción, alcance, datos causales y versiones

- **Alcance exclusivo:** 17 plantas fotovoltaicas de Colombia (registros XM y pronósticos meteorológicos Open-Meteo / GEFSv12) organizadas en **28 escenarios oficiales** (`4 D+1`, `8 Recursivo sin clima`, `8 Directo sin clima`, `8 Recursivo + GEFSv12`).
- **Identificadores de versión:** paquete `sm-cfhr 1.0.3` (`pyproject.toml` y `sm_cfhr.__version__`), licencia `PolyForm-Noncommercial-1.0.0`, etiqueta `v1.0.3-polyform`, protocolo interno `v1.0.3-exact-recursive-state`, commit base OOF `3256994`, semilla `2026`.
- **Nota de alcance académico:** No se incorporan datasets externos de otros dominios; la sustitución por los 28 escenarios solares queda documentada como modificación de alcance pendiente de validación docente explícita.

In [1]:
import json
import sys
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

_CUR = Path.cwd().resolve()
ROOT = next(p for p in (_CUR, *_CUR.parents) if (p / "experiments" / "config_ashr_congelada.json").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import sm_cfhr
from sm_cfhr import (
    AdaptiveStructuredHuberRegressor,
    SCENARIOS_28,
    SMCFHRRegressor,
    analyze_effective_design,
    evaluate_r3_superiority,
    export_full_scientific_artifacts,
    summarize_hpo_trials,
    verify_benchmark_inventory,
)
from sm_cfhr.plotting import (
    export_all_figures,
    plot_ablation_bar,
    plot_conformal_coverage,
    plot_feature_importance,
    plot_hp_sensitivity,
    plot_learning_and_scalability,
    plot_nested_cv_scheme,
    plot_optuna_convergence,
    plot_robustness_stress,
)
from sm_cfhr.pipeline_stages import apply_family_multiplicity
from src.ashr_oof_pipeline import (
    build_descriptive_summary,
    export_paper_tables,
    load_frozen_config,
    load_oof_bundle,
    validate_oof_artifacts,
)
from src.ashr_oof_pipeline.oof_runner import run_oof_definitivo

CFG = load_frozen_config(ROOT)
CONTRATO_RUTAS = CFG.get("rutas_contrato", {})
OOF_DIR = ROOT / CONTRATO_RUTAS.get("oof_dir", "resultados/evaluacion_ashr_oof")
HPO_TRIALS_DIR = ROOT / CONTRATO_RUTAS.get("hpo_trials_dir", "resultados/evaluacion_ashr_oof/hpo_trials")
FIG_DIR = OOF_DIR / "figuras"
TAB_DIR = OOF_DIR / "tablas_paper"
SCOPE_AUTHORIZED = bool(CFG.get("autorizacion_alcance_solar_docente", False))

print("Paquete sm_cfhr versión:", sm_cfhr.__version__)
print("Denominación científica:", CFG["denominacion_cientifica"])
print("Protocolo HPO congelado:", "v1.0.3-exact-recursive-state")
print("SHA-256 núcleo congelado:", CFG["sha256_src_adaptive_structured_huber"])

C:\Users\Enma\miniconda3\envs\solarpv-eda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Paquete sm_cfhr versión: 1.0.3
Denominación científica: SM-CFHR (Structured Monotone Capacity-Factor Huber Regressor) / ASHR-Fixed
Protocolo HPO congelado: v1.0.3-exact-recursive-state
SHA-256 núcleo congelado: 27a9fda4fa8b7ef0d95faf9e925e47ade9ed8f25e9ea5d97d6d7cf1c00ddde83


## 2. Revisión de literatura y novedad acotada (R1)

El protocolo formal de literatura (`docs/auditoria_entregable3/AUDITORIA_R1_R2_R3.md`) documenta 24 exportaciones de **Web of Science Core Collection** (`fuentes/bibliografia/wos/`), análisis de 8 trabajos cercanos (Pico et al. 2026; Trigo-González et al. 2019; Almeida et al. 2015; Yang et al. 2020; Huber 1964; Owen 2007; K1/W1) y *snowballing*.

**Afirmación acotada de novedad:** Formulación paramétrica convexa propia en espacio adimensional de factor de capacidad (`CF`) que combina pérdida Huber con escala MAD, regularización $L_2$ diferenciada por familia y restricciones físicas monótonas (`cf30 >= 0`, `rad_pron >= 0`), optimizada con L-BFGS-B analítico propio en NumPy/SciPy.

## 3. Formulación matemática y propiedades de `SM-CFHR`

Sea $\mathrm{cap}_i > 0$ la capacidad instalada previa al origen y $B_i = \mathrm{cap}_i \cdot 24{,}000\;\text{kWh/día}$. En diarios/recursivos $c_i = \operatorname{clip}(y_i/B_i, 0, 1)$; en directos acumulados $c_i = \operatorname{clip}(Y_{H,i}/(H B_i), 0, 1)$.

Con respuesta robusta $\tilde{c}_i = (c_i - m_c)/\hat{\sigma}_c$ ($\hat{\sigma}_c = \max(1.4826\,\mathrm{MAD}(c), 10^{-6})$) y predictores estandarizados $z_i$:

$$\min_{\theta_0 \in \mathbb{R},\; \theta \in \mathcal{C}} \; \frac{1}{n}\sum_{i=1}^n \rho_\delta\!\left(\tilde{c}_i - \theta_0 - z_i^\top \theta\right) + \frac{1}{2}\sum_{j \in \mathcal{G}_{\mathrm{lin}}} \alpha_{\mathrm{lin}}\,\theta_j^2 + \frac{1}{2}\sum_{j \in \mathcal{G}_{\mathrm{meteo}}} \alpha_{\mathrm{meteo}}\,\theta_j^2$$

sujeto a $\mathcal{C} = \{\theta : \theta_j \ge 0 \;\forall j \in \mathcal{M}_{\mathrm{pos}}\}$.

## 4. Paquete `sm_cfhr` e integración `scikit-learn` (R2)

`SMCFHRRegressor` y `AdaptiveStructuredHuberRegressor` heredan de `(BaseEstimator, RegressorMixin)` sin delegar el algoritmo en estimadores de `scikit-learn`. Se verifica `clone`, `Pipeline` y `get_params` sin ejecutar entrenamiento OOF.

In [2]:
hp_cfg = CFG["parametros_modelo"]["fixed_config"]
sklearn_demo_model = SMCFHRRegressor(
    delta=CFG["parametros_modelo"]["delta"],
    alpha_lin=hp_cfg["alpha_lin"],
    alpha_meteo=hp_cfg["alpha_meteo"],
    structured=hp_cfg["structured"],
    use_mono=hp_cfg["use_mono"],
    random_state=CFG["parametros_modelo"]["random_state"],
)
cloned_demo = clone(sklearn_demo_model)
pipe_demo = Pipeline([("scaler", StandardScaler()), ("sm_cfhr", cloned_demo)])

print("Es BaseEstimator:", isinstance(sklearn_demo_model, BaseEstimator))
print("Es RegressorMixin:", isinstance(sklearn_demo_model, RegressorMixin))
print("get_params == clone.get_params:", sklearn_demo_model.get_params() == cloned_demo.get_params())
print("Pasos de Pipeline:", [name for name, _ in pipe_demo.steps])

Es BaseEstimator: True
Es RegressorMixin: True
get_params == clone.get_params: True
Pasos de Pipeline: ['scaler', 'sm_cfhr']


## 5. Optimización bayesiana Optuna TPE y auditoría de estudios

Se comparan dos variantes del mismo núcleo fijo (`n_knots=0, max_interactions=0, anchor="none", structured=True, use_mono=True`):
- **`SM-CFHR-Fixed` ($B=0$):** `delta=1.35, alpha_lin=0.01, alpha_meteo=0.10`.
- **`SM-CFHR-HPO` ($B=25$):** Optuna `TPESampler` sobre `delta ∈ [0.5, 3.0]`, `alpha_lin ∈ [1e-4, 1.0]` (log) y `alpha_meteo ∈ [1e-3, 10.0]` (log, vinculado a `alpha_lin` si no hay meteorología), optimizando `MAE_macro_kWh` dentro de 3 particiones causales internas más 1 *refit* final por unidad.

**Nota de auditoría de estudios:** cada estudio Optuna queda unívocamente identificado por `(scenario_id, outer_fold)`. De las `14,625` unidades HPO totales, `14,552` ejecutaron estudio Optuna (`14,544` con `25` filas y `25` trials válidos; `8` unidades reanudadas registraron `26` filas con `1` trial fallido/interrumpido y exactamente `25` trials `COMPLETE` válidos), mientras que `73` unidades tempranas sin historial suficiente usaron la configuración por defecto sin HPO.

In [3]:
hpo_space_df = pd.DataFrame([
    {"hiperparametro": "delta", "escala": "uniforme", "min": 0.5, "max": 3.0, "valor_fixed": CFG["parametros_modelo"]["delta"]},
    {"hiperparametro": "alpha_lin", "escala": "log-uniforme", "min": 1e-4, "max": 1.0, "valor_fixed": hp_cfg["alpha_lin"]},
    {"hiperparametro": "alpha_meteo", "escala": "log-uniforme (si weather_on)", "min": 1e-3, "max": 10.0, "valor_fixed": hp_cfg["alpha_meteo"]},
])
display(hpo_space_df)

hpo_trials_df = pd.DataFrame()
if HPO_TRIALS_DIR.exists():
    trial_files = sorted(HPO_TRIALS_DIR.glob("*.csv"))
    if trial_files:
        hpo_trials_df = pd.concat([pd.read_csv(f).assign(scenario_id=f.stem) for f in trial_files], ignore_index=True)
        hpo_summary_df = summarize_hpo_trials(hpo_trials_df)
        print(
            f"Estudios Optuna únicos (scenario_id, outer_fold): {len(hpo_summary_df):,} | "
            f"Trials válidos totales: {int(hpo_summary_df['n_valid_trials'].sum()):,} | "
            f"Min/Max trials válidos por estudio: {int(hpo_summary_df['n_valid_trials'].min())}/{int(hpo_summary_df['n_valid_trials'].max())}"
        )
        hpo_by_scenario = (
            hpo_summary_df.groupby("scenario_id", as_index=False)
            .agg(
                n_estudios=("outer_fold", "count"),
                trials_validos_por_estudio_min=("n_valid_trials", "min"),
                trials_validos_por_estudio_max=("n_valid_trials", "max"),
                mediana_delta=("best_delta", "median"),
                mediana_alpha_lin=("best_alpha_lin", "median"),
                mediana_alpha_meteo=("best_alpha_meteo", "median"),
                tiempo_total_trials_s=("total_wall_time_s", "sum"),
            )
        )
        display(hpo_by_scenario)
        plot_optuna_convergence(hpo_trials_df, FIG_DIR)
else:
    print("Logs de trials Optuna TPE: PENDIENTE DE EJECUCIÓN OOF HPO.")

,hiperparametro,escala,min,max,valor_fixed
0,delta,uniforme,0.5000,3.0,1.35
1,alpha_lin,log-uniforme,0.0001,1.0,0.01
2,alpha_meteo,log-uniforme (si weather_on),0.0010,10.0,0.10


Estudios Optuna únicos (scenario_id, outer_fold): 14,552 | Trials válidos totales: 363,800 | Min/Max trials válidos por estudio: 25/25


,scenario_id,n_estudios,trials_validos_por_estudio_min,trials_validos_por_estudio_max,mediana_delta,mediana_alpha_lin,mediana_alpha_meteo,tiempo_total_trials_s
0,D1_espaciotemporal_cinco_variables,20,25,25,1.702129,0.010525,0.003828,42.941217
1,D1_espaciotemporal_sin_meteorologia,20,25,25,1.825299,0.208786,0.208786,27.890127
2,D1_intraplanta_cinco_variables,49,25,25,1.899277,0.332529,0.187023,91.527296
3,D1_intraplanta_sin_meteorologia,49,25,25,1.492608,0.436930,0.436930,50.642964
4,DirectoSinClima_espaciotemporal_14d,356,25,25,0.722895,0.091214,0.091214,280.059895
5,DirectoSinClima_espaciotemporal_21d,229,25,25,1.053403,0.066875,0.066875,182.390456
6,DirectoSinClima_espaciotemporal_28d,168,25,25,1.250596,0.137144,0.137144,127.379992
7,DirectoSinClima_espaciotemporal_7d,730,25,25,0.885109,0.058759,0.058759,751.011840
8,DirectoSinClima_intraplanta_14d,779,25,25,1.250596,0.882082,0.882082,676.641456
9,DirectoSinClima_intraplanta_21d,481,25,25,1.250596,0.907754,0.907754,290.368217


## 6. Validación cruzada causal anidada temporal y espacial

- **D+1 (`d1`):** ventana creciente con embargo (`30d` cuando el historial lo permite), $\max(\text{date}_{\text{train}}) < \min(\text{date}_{\text{val}})$.
- **Directo (`direct`):** $\max(\text{t\_end\_target}_{\text{train}}) \le \min(\text{t\_origin}_{\text{val}})$.
- **Recursivo (`recursive`):** $\max(\text{origin}_{\text{train}}) + H \le \min(\text{origin}_{\text{val}})$.
- **Espaciotemporal (`ET`):** retención de grupo espacial completo en validación interna (`spatial_disjoint=True`).

La tabla siguiente proviene de `particiones/nested_cv_audit.csv`, reconstruida directamente sobre los paneles reales mediante `build_causal_inner_splits` y `audit_inner_splits` para folds representativos de las 4 suites (en ET e IP).

In [4]:
nested_audit_path = OOF_DIR / "particiones" / "nested_cv_audit.csv"
nested_audit_df = pd.DataFrame()
if nested_audit_path.exists():
    nested_audit_df = pd.read_csv(nested_audit_path)
    print(
        f"Auditoría Nested CV: {len(nested_audit_df)} splits representativos | "
        f"Gap efectivo mínimo: {int(nested_audit_df['effective_gap_days'].min())} días | "
        f"Espacialmente disjuntos (ET): {bool(nested_audit_df.loc[nested_audit_df['task']=='espaciotemporal', 'spatial_disjoint'].all())}"
    )
    display(nested_audit_df)
    plot_nested_cv_scheme(nested_audit_df, FIG_DIR)
else:
    print("Trazabilidad de folds de Nested CV definitiva: PENDIENTE DE GENERACIÓN DE AUDITORÍA.")

Auditoría Nested CV: 18 splits representativos | Gap efectivo mínimo: 7 días | Espacialmente disjuntos (ET): True


,split_idx,mode_eval,task,horizon_days,train_min_date,train_max_date,train_max_target_end,val_min_date,val_max_date,effective_gap_days,train_groups,val_groups,spatial_disjoint,n_train,n_val,n_plants_train,n_plants_val,scenario_id,outer_fold_id
0,0,d1,espaciotemporal,1,2024-03-22,2025-12-25,2025-12-25,2026-01-24,2026-06-01,30,"2,3,4,5,6",1,True,6111,236,14,2,D1_espaciotemporal_cinco_variables,representative
1,1,d1,espaciotemporal,1,2024-05-22,2025-08-29,2025-08-29,2025-09-28,2026-01-23,30,"1,3,4,5,6",2,True,4250,236,14,2,D1_espaciotemporal_cinco_variables,representative
2,2,d1,espaciotemporal,1,2024-03-22,2025-05-03,2025-05-03,2025-06-02,2025-09-27,30,"1,2,4,5,6",3,True,2950,234,13,2,D1_espaciotemporal_cinco_variables,representative
3,0,d1,intraplanta,1,2024-09-07,2026-01-20,2026-01-20,2026-02-19,2026-06-01,30,5,5,False,496,92,1,1,D1_intraplanta_cinco_variables,representative
4,1,d1,intraplanta,1,2024-09-07,2025-10-20,2025-10-20,2025-11-19,2026-02-18,30,5,5,False,404,92,1,1,D1_intraplanta_cinco_variables,representative
5,2,d1,intraplanta,1,2024-09-07,2025-07-20,2025-07-20,2025-08-19,2025-11-18,30,5,5,False,312,92,1,1,D1_intraplanta_cinco_variables,representative
6,0,recursive,espaciotemporal,7,2024-01-16,2026-05-14,2026-05-21,2026-05-21,2026-10-17,7,"2,3,4,5,6,7",1,True,8269,299,15,2,RecSinClima_espaciotemporal_7d,representative
7,1,recursive,espaciotemporal,7,2024-05-22,2025-12-15,2025-12-22,2025-12-22,2026-05-20,7,"1,3,4,5,6",2,True,5716,298,14,2,RecSinClima_espaciotemporal_7d,representative
8,2,recursive,espaciotemporal,7,2024-01-16,2025-07-18,2025-07-25,2025-07-25,2025-12-21,7,"1,2,4,5,6",3,True,4061,297,14,2,RecSinClima_espaciotemporal_7d,representative
9,0,recursive,intraplanta,7,2024-09-07,2026-06-19,2026-06-26,2026-06-26,2026-10-17,7,5,5,False,644,114,1,1,RecGEFSv12_intraplanta_7d,representative


## 7. Protocolo experimental e inventario de 28 escenarios y 22 benchmarks

La comparación principal conservadora enfrenta `SM-CFHR-Fixed` ($B=0$) contra los **22 modelos oficiales** (`9` librerías, `12` custom, `MM30`; `CF-BRH` y `R-ILWR-PA v3` excluidos). `SM-CFHR-HPO` ($B=25$) se reporta en paralelo declarando la asimetría frente a rejillas de 4 puntos.

In [5]:
inv_df = verify_benchmark_inventory(ROOT)
print(f"Escenarios verificados: {len(inv_df)}/28 | Archivos OK: {inv_df['file_exists'].all()} | Columnas OK: {inv_df['cols_ok'].all()} | CF-BRH excluido: {inv_df['cf_brh_excluded'].all()}")
display(inv_df)

Escenarios verificados: 28/28 | Archivos OK: True | Columnas OK: True | CF-BRH excluido: True


,scenario_id,suite,task,horizon_days,weather_on,mode_eval,unit,file_exists,cols_ok,n_official_models,cf_brh_excluded
0,D1_espaciotemporal_cinco_variables,D+1 (37-40),espaciotemporal,1,True,d1,kWh/dia,True,True,22,True
1,D1_espaciotemporal_sin_meteorologia,D+1 (37-40),espaciotemporal,1,False,d1,kWh/dia,True,True,22,True
2,D1_intraplanta_cinco_variables,D+1 (37-40),intraplanta,1,True,d1,kWh/dia,True,True,22,True
3,D1_intraplanta_sin_meteorologia,D+1 (37-40),intraplanta,1,False,d1,kWh/dia,True,True,22,True
4,RecSinClima_espaciotemporal_7d,Rec sin clima (43-46),espaciotemporal,7,False,recursive,kWh/dia,True,True,22,True
5,RecSinClima_intraplanta_7d,Rec sin clima (43-46),intraplanta,7,False,recursive,kWh/dia,True,True,22,True
6,RecSinClima_espaciotemporal_14d,Rec sin clima (43-46),espaciotemporal,14,False,recursive,kWh/dia,True,True,22,True
7,RecSinClima_intraplanta_14d,Rec sin clima (43-46),intraplanta,14,False,recursive,kWh/dia,True,True,22,True
8,RecSinClima_espaciotemporal_21d,Rec sin clima (43-46),espaciotemporal,21,False,recursive,kWh/dia,True,True,22,True
9,RecSinClima_intraplanta_21d,Rec sin clima (43-46),intraplanta,21,False,recursive,kWh/dia,True,True,22,True


## 8. Detección de estado, integridad SHA-256 y auditoría de alineación

Esta celda verifica manifiesto, hashes SHA-256 y la alineación exacta de claves `(code, date/origen/lead_dias)` entre `SM-CFHR-Fixed`, `SM-CFHR-HPO` y los 22 benchmarks oficiales. En `D1` y `Recursivo`, los orígenes extra son `0`; en los 8 escenarios `DirectoSinClima`, `ashr_df` contenía `2`–`63` orígenes tempranos (`len(train) >= 30`) fuera del soporte del CSV oficial del benchmark, por lo que `load_oof_bundle` intersecta sobre las claves comunes exactas del benchmark antes de evaluar.

In [6]:
dry_status = run_oof_definitivo(execute=False, root=ROOT)
val_status = validate_oof_artifacts(OOF_DIR, CFG["sha256_src_adaptive_structured_huber"])
state = val_status["state"]
print("Estado detectado del notebook:", state)
for rsn in val_status["reasons"]:
    print(" -", rsn)

bundle = None
hpo_preds_available = False
if state == "VALID":
    bundle = load_oof_bundle(OOF_DIR, root=ROOT)
    hpo_files = list((OOF_DIR / "predicciones").glob("*_hpo.csv"))
    hpo_preds_available = len(hpo_files) == len(SCENARIOS_28)
    print(f"Estado B ACTIVO — 28 escenarios validados y alineados (HPO disponible: {hpo_preds_available}).")
    align_path = OOF_DIR / "inferencia" / "auditoria_alineacion_benchmarks.csv"
    if align_path.exists():
        align_df = pd.read_csv(align_path)
        print(
            f"Claves Fixed == HPO en 28/28 escenarios: {bool(align_df['fixed_hpo_keys_identical'].all())} | "
            f"Escenarios con orígenes extra excluidos: {int((align_df['n_extra_excluded'] > 0).sum())}/28 (todos DirectoSinClima)"
        )
        display(align_df)
elif state == "INVALID":
    print("ESTADO C (INCONSISTENTE / PARCIAL): se bloquean conclusiones confirmatorias.")
else:
    print("ESTADO A (PENDIENTE): ejecutar OOF definitivo antes de analizar.")

Estado detectado del notebook: VALID


Estado B ACTIVO — 28 escenarios validados y alineados (HPO disponible: True).
Claves Fixed == HPO en 28/28 escenarios: True | Escenarios con orígenes extra excluidos: 8/28 (todos DirectoSinClima)


,scenario_id,suite,n_ashr_raw,n_bench_keys,n_extra_excluded,n_missing_in_ashr,n_aligned_common,fixed_hpo_keys_identical,mae_macro_fixed_raw,mae_macro_fixed_aligned
0,D1_espaciotemporal_cinco_variables,D+1 (37-40),4312,4312,0,0,4312,True,5.980365e+04,5.980365e+04
1,D1_espaciotemporal_sin_meteorologia,D+1 (37-40),4312,4312,0,0,4312,True,7.781063e+04,7.781063e+04
2,D1_intraplanta_cinco_variables,D+1 (37-40),4269,4269,0,0,4269,True,6.129709e+04,6.129709e+04
3,D1_intraplanta_sin_meteorologia,D+1 (37-40),4269,4269,0,0,4269,True,7.816278e+04,7.816278e+04
4,RecSinClima_espaciotemporal_7d,Rec sin clima (43-46),11993,12061,0,68,11993,True,8.340880e+04,8.340880e+04
5,RecSinClima_intraplanta_7d,Rec sin clima (43-46),11283,11347,0,64,11283,True,8.356065e+04,8.356065e+04
6,RecSinClima_espaciotemporal_14d,Rec sin clima (43-46),11909,11984,0,75,11909,True,8.398637e+04,8.398637e+04
7,RecSinClima_intraplanta_14d,Rec sin clima (43-46),11255,11326,0,71,11255,True,8.361428e+04,8.361428e+04
8,RecSinClima_espaciotemporal_21d,Rec sin clima (43-46),11587,11676,0,89,11587,True,8.503677e+04,8.503677e+04
9,RecSinClima_intraplanta_21d,Rec sin clima (43-46),10941,11025,0,84,10941,True,8.402051e+04,8.402051e+04


## 9. Resultados OOF globales y comparación `SM-CFHR-Fixed` vs. `SM-CFHR-HPO`

- La métrica confirmatoria oficial es **`MAE_macro`** (media por planta `code` y luego promedio entre plantas, en `kWh/dia` para `D+1`/recursivo y `kWh_total` para directo).
- Se reporta **`MAE_micro`** (media global por observación) únicamente como métrica secundaria explícita. Sobre el soporte común alineado, **ambas métricas (`MAE_macro` y `MAE_micro`) coinciden en `9 / 28` victorias de `HPO` sobre `Fixed`** (todas en escenarios `intraplanta`, mientras `Fixed` gana en `14/14` escenarios `espaciotemporal` y `5/14` `intraplanta`). La discrepancia previa de 10 victorias micro en un borrador anterior provenía de incluir `63` orígenes tempranos no alineados en `DirectoSinClima_intraplanta_7d`.

In [7]:
if bundle is not None:
    print("=== Resumen por régimen y tarea (SM-CFHR-Fixed) ===")
    display(bundle["tabla_rankings_regimen"])
    print("=== Resultados completos por escenario (SM-CFHR-Fixed) ===")
    display(bundle["tabla_28_escenarios"])
    fh_path = OOF_DIR / "metricas" / "comparacion_fixed_vs_hpo.csv"
    if fh_path.exists():
        fh_df = pd.read_csv(fh_path)
        print(
            f"=== Comparación Fixed (B=0) vs HPO (B=25) sobre soporte alineado ===\n"
            f"Victorias HPO vs Fixed (MAE_macro confirmatorio): {int((fh_df['diff_macro_HPO_minus_Fixed'] < 0).sum())}/28 | "
            f"(MAE_micro secundario): {int((fh_df['diff_micro_HPO_minus_Fixed'] < 0).sum())}/28\n"
            f"Rango medio en 23 — Fixed: {fh_df['Rank_Fixed_en_23'].mean():.2f} | HPO: {fh_df['Rank_HPO_en_23'].mean():.2f}"
        )
        display(fh_df)
    export_all_figures(bundle, FIG_DIR)
else:
    print("Tablas OOF completas y figuras 1–5: PENDIENTE DE ESTADO B.")

=== Resumen por régimen y tarea (SM-CFHR-Fixed) ===


,suite,task,n_escenarios,rango_medio,victorias_vs_bib,victorias_vs_custom,victorias_vs_mm30,pct_medio_vs_bib
0,D+1 (37-40),espaciotemporal,2,1.00,2,2,2,-11.300467
1,D+1 (37-40),intraplanta,2,7.50,1,0,2,0.068939
2,Directo sin clima (48-51),espaciotemporal,4,3.25,3,3,0,-1.629684
3,Directo sin clima (48-51),intraplanta,4,6.50,1,1,0,13.268070
4,Rec con clima (53-56),espaciotemporal,4,1.00,4,4,4,-11.841720
5,Rec con clima (53-56),intraplanta,4,2.00,0,4,4,1.037242
6,Rec sin clima (43-46),espaciotemporal,4,2.00,4,4,0,-1.697857
7,Rec sin clima (43-46),intraplanta,4,1.50,3,4,3,-0.269327


=== Resultados completos por escenario (SM-CFHR-Fixed) ===


,scenario_id,suite,task,horizon_days,weather_on,mode_eval,unit,n_obs,MAE_ASHR,RMSE_ASHR,...,Mejor_Biblioteca,MAE_Mejor_Bib,pct_vs_MejorLib,Mejor_Custom,MAE_Mejor_Custom,pct_vs_MejorCustom,MAE_MM30,Skill_vs_MM30_pct,rango_en_23,n_modelos_comparados
0,D1_espaciotemporal_cinco_variables,D+1 (37-40),espaciotemporal,1,True,d1,kWh/dia,4312,5.980365e+04,1.176746e+05,...,Lasso,7.683088e+04,-22.161971,RQRF,7.600882e+04,-21.320123,8.399214e+04,28.798520,1,23
1,D1_espaciotemporal_sin_meteorologia,D+1 (37-40),espaciotemporal,1,False,d1,kWh/dia,4312,7.781063e+04,1.490129e+05,...,Lasso,7.815369e+04,-0.438962,KLAC,7.798614e+04,-0.225056,8.399214e+04,7.359630,1,23
2,D1_intraplanta_cinco_variables,D+1 (37-40),intraplanta,1,True,d1,kWh/dia,4269,6.129709e+04,1.198765e+05,...,BayesianRidge,6.143523e+04,-0.224856,WCFC,5.745753e+04,6.682421,8.406497e+04,27.083677,12,23
3,D1_intraplanta_sin_meteorologia,D+1 (37-40),intraplanta,1,False,d1,kWh/dia,4269,7.816278e+04,1.478835e+05,...,BayesianRidge,7.788028e+04,0.362734,ELAC,7.773085e+04,0.555678,8.406497e+04,7.020992,3,23
4,RecSinClima_espaciotemporal_7d,Rec sin clima (43-46),espaciotemporal,7,False,recursive,kWh/dia,11993,8.340880e+04,1.537638e+05,...,BayesianRidge,8.437169e+04,-1.141246,RARC,8.493745e+04,-1.799732,8.272046e+04,-0.832125,2,23
5,RecSinClima_intraplanta_7d,Rec sin clima (43-46),intraplanta,7,False,recursive,kWh/dia,11283,8.356065e+04,1.531796e+05,...,SVR,8.328076e+04,0.336088,ELAC,8.552298e+04,-2.294505,8.366420e+04,0.123771,2,23
6,RecSinClima_espaciotemporal_14d,Rec sin clima (43-46),espaciotemporal,14,False,recursive,kWh/dia,11909,8.398637e+04,1.544066e+05,...,BayesianRidge,8.517960e+04,-1.400842,RARC,8.567299e+04,-1.968680,8.300436e+04,-1.183077,2,23
7,RecSinClima_intraplanta_14d,Rec sin clima (43-46),intraplanta,14,False,recursive,kWh/dia,11255,8.361428e+04,1.531127e+05,...,BayesianRidge,8.402525e+04,-0.489102,ELAC,8.675919e+04,-3.624873,8.399144e+04,0.449037,1,23
8,RecSinClima_espaciotemporal_21d,Rec sin clima (43-46),espaciotemporal,21,False,recursive,kWh/dia,11587,8.503677e+04,1.557491e+05,...,BayesianRidge,8.644151e+04,-1.625077,RARC,8.600696e+04,-1.128042,8.362077e+04,-1.693356,2,23
9,RecSinClima_intraplanta_21d,Rec sin clima (43-46),intraplanta,21,False,recursive,kWh/dia,10941,8.402051e+04,1.542229e+05,...,BayesianRidge,8.480302e+04,-0.922744,SRMOS,8.749756e+04,-3.973889,8.427810e+04,0.305647,1,23


=== Comparación Fixed (B=0) vs HPO (B=25) sobre soporte alineado ===
Victorias HPO vs Fixed (MAE_macro confirmatorio): 9/28 | (MAE_micro secundario): 9/28
Rango medio en 23 — Fixed: 2.93 | HPO: 3.86


,scenario_id,suite,task,horizon_days,unit,n_obs,MAE_macro_Fixed,MAE_macro_HPO,diff_macro_HPO_minus_Fixed,rel_macro_HPO_vs_Fixed_pct,...,MAE_micro_HPO,diff_micro_HPO_minus_Fixed,Skill_Fixed_vs_MM30_pct,Skill_HPO_vs_MM30_pct,pct_Fixed_vs_BestBib,pct_HPO_vs_BestBib,Rank_Fixed_en_23,Rank_HPO_en_23,p_dm_hln_hpo_vs_fixed,T_eff_hpo_vs_fixed
0,D1_espaciotemporal_cinco_variables,D+1 (37-40),espaciotemporal,1,kWh/dia,4312,5.980365e+04,6.086148e+04,1057.834570,1.768846,...,6.229556e+04,1096.136589,28.798520,27.539075,-22.161971,-20.785136,1,1,0.058852,183.23
1,D1_espaciotemporal_sin_meteorologia,D+1 (37-40),espaciotemporal,1,kWh/dia,4312,7.781063e+04,7.982807e+04,2017.438373,2.592754,...,8.173476e+04,2079.843770,7.359630,4.957693,-0.438962,2.142411,1,8,0.000027,178.69
2,D1_intraplanta_cinco_variables,D+1 (37-40),intraplanta,1,kWh/dia,4269,6.129709e+04,5.974116e+04,-1555.925323,-2.538335,...,6.160265e+04,-1053.919496,27.083677,28.934537,-0.224856,-2.757483,12,10,0.000013,213.14
3,D1_intraplanta_sin_meteorologia,D+1 (37-40),intraplanta,1,kWh/dia,4269,7.816278e+04,7.950643e+04,1343.652714,1.719044,...,8.193321e+04,2483.497071,7.020992,5.422642,0.362734,2.088014,3,6,0.000166,166.43
4,RecSinClima_espaciotemporal_7d,Rec sin clima (43-46),espaciotemporal,7,kWh/dia,11993,8.340880e+04,8.481094e+04,1402.137416,1.681043,...,8.630360e+04,1484.229201,-0.832125,-2.527156,-1.141246,0.520612,2,4,0.307367,68.06
5,RecSinClima_intraplanta_7d,Rec sin clima (43-46),intraplanta,7,kWh/dia,11283,8.356065e+04,8.347768e+04,-82.971827,-0.099295,...,8.326589e+04,-168.507210,0.123771,0.222944,0.336088,0.236459,2,2,0.086054,113.79
6,RecSinClima_espaciotemporal_14d,Rec sin clima (43-46),espaciotemporal,14,kWh/dia,11909,8.398637e+04,8.548156e+04,1495.196290,1.780285,...,8.727828e+04,1608.045741,-1.183077,-2.984424,-1.400842,0.354503,2,4,0.240240,41.59
7,RecSinClima_intraplanta_14d,Rec sin clima (43-46),intraplanta,14,kWh/dia,11255,8.361428e+04,8.316758e+04,-446.703179,-0.534243,...,8.312412e+04,-438.656229,0.449037,0.980881,-0.489102,-1.020731,1,1,0.101833,46.33
8,RecSinClima_espaciotemporal_21d,Rec sin clima (43-46),espaciotemporal,21,kWh/dia,11587,8.503677e+04,8.618371e+04,1146.942210,1.348760,...,8.771658e+04,1172.446887,-1.693356,-3.064955,-1.625077,-0.298235,2,3,0.921715,27.10
9,RecSinClima_intraplanta_21d,Rec sin clima (43-46),intraplanta,21,kWh/dia,10941,8.402051e+04,8.332751e+04,-692.990966,-0.824788,...,8.335713e+04,-597.013506,0.305647,1.127914,-0.922744,-1.739922,1,1,0.290911,39.50


## 10. Inferencia estadística (DM-HLN-HAC, MCS, SPA post-hoc y multiplicidad)

- `DM-HLN-HAC` y `MCS` ($lpha=0{,}10$, $B=200$) integran la evaluación automática del flujo OOF sobre series temporales agregadas por fecha/origen.
- `SPA` ($B=500$, longitud media de bloque $7$, [tabla_spa.csv](inferencia/tabla_spa.csv)) se ejecuta como análisis complementario post-hoc bajo $H_0$: `MM30` no es superada y bajo $H_0$: `ASHR` no es superado.
- **Cautela interpretativa:** ganar descriptivamente (`18/28` vs mejor biblioteca) o permanecer en el MCS (`27/28`) **no** equivale a demostrar superioridad estadística general cuando solo `2/15` escenarios con $T_{\text{eff}} \ge 2H$ alcanzan significancia tras Holm–Bonferroni.

In [8]:
if bundle is not None:
    t_inf = apply_family_multiplicity(bundle["tabla_inferencia"])
    t_mcs = bundle["tabla_mcs"].copy()
    print("=== Inferencia pareada DM-HLN-HAC + Holm/BH por familia ===")
    display(t_inf)
    print(f"=== Model Confidence Set (alpha=0.10): ASHR sobrevive en {int(t_mcs['ashr_in_mcs_90'].sum())}/{len(t_mcs)} escenarios ===")
    display(t_mcs)
    spa_path = OOF_DIR / "inferencia" / "tabla_spa.csv"
    if spa_path.exists():
        spa_df = pd.read_csv(spa_path)
        print("=== Prueba global SPA (post-hoc, B=500) ===")
        display(spa_df)
else:
    print("Inferencia estadística y tabla MCS: PENDIENTE DE ESTADO B.")

=== Inferencia pareada DM-HLN-HAC + Holm/BH por familia ===


,escenario,suite,task,horizon_days,unit,referencia,modelo_referencia,T_eff,t_eff_insufficient,mean_diff,...,cohen_dz_pareado,r_rangobiserial_pareado,prob_victoria_A,cliffs_delta_marginal,p_holm,estado_estadistico,familia_id,p_holm_familia,p_bh_familia,n_comparaciones_familia
0,D1_espaciotemporal_cinco_variables,D+1 (37-40),espaciotemporal,1,kWh/dia,Mejor_Bib,Lasso,152.56,False,-16920.826527,...,-0.898,-0.832,0.817,-0.407,6.390590e-22,SIGNIFICATIVO_HOLM,D+1 (37-40)__Mejor_Bib,4.564707e-23,4.564707e-23,4
1,D1_espaciotemporal_cinco_variables,D+1 (37-40),espaciotemporal,1,kWh/dia,MM30,Media móvil de 30 días,137.81,False,-24842.917271,...,-0.754,-0.735,0.752,-0.425,7.236091e-15,SIGNIFICATIVO_HOLM,D+1 (37-40)__MM30,5.262611e-16,5.262611e-16,4
2,D1_espaciotemporal_sin_meteorologia,D+1 (37-40),espaciotemporal,1,kWh/dia,Mejor_Bib,Lasso,180.06,False,-316.573064,...,-0.071,-0.184,0.584,-0.019,1.000000e+00,NO_SIGNIFICATIVO,D+1 (37-40)__Mejor_Bib,6.375496e-01,3.430626e-01,4
3,D1_espaciotemporal_sin_meteorologia,D+1 (37-40),espaciotemporal,1,kWh/dia,MM30,Media móvil de 30 días,131.23,False,-6374.130363,...,-0.373,-0.388,0.637,-0.115,1.405563e-03,SIGNIFICATIVO_HOLM,D+1 (37-40)__MM30,5.406011e-05,3.604008e-05,4
4,D1_intraplanta_cinco_variables,D+1 (37-40),intraplanta,1,kWh/dia,Mejor_Bib,BayesianRidge,116.59,False,-458.727847,...,-0.143,-0.165,0.573,-0.016,1.000000e+00,NO_SIGNIFICATIVO,D+1 (37-40)__Mejor_Bib,3.751754e-01,2.501170e-01,4
5,D1_intraplanta_cinco_variables,D+1 (37-40),intraplanta,1,kWh/dia,MM30,Media móvil de 30 días,132.93,False,-23885.204821,...,-0.687,-0.672,0.752,-0.403,3.672709e-12,SIGNIFICATIVO_HOLM,D+1 (37-40)__MM30,2.040394e-13,1.360262e-13,4
6,D1_intraplanta_sin_meteorologia,D+1 (37-40),intraplanta,1,kWh/dia,Mejor_Bib,BayesianRidge,195.27,False,-243.840828,...,-0.071,-0.111,0.573,-0.011,1.000000e+00,NO_SIGNIFICATIVO,D+1 (37-40)__Mejor_Bib,6.375496e-01,3.430626e-01,4
7,D1_intraplanta_sin_meteorologia,D+1 (37-40),intraplanta,1,kWh/dia,MM30,Media móvil de 30 días,147.85,False,-7247.370419,...,-0.335,-0.347,0.611,-0.131,3.187185e-03,SIGNIFICATIVO_HOLM,D+1 (37-40)__MM30,6.249383e-05,6.249383e-05,4
8,RecSinClima_espaciotemporal_7d,Rec sin clima (43-46),espaciotemporal,7,kWh/dia,Mejor_Bib,BayesianRidge,71.54,False,-1123.967101,...,-0.091,-0.343,0.667,-0.050,1.000000e+00,NO_SIGNIFICATIVO,Rec sin clima (43-46)__Mejor_Bib,1.000000e+00,6.231130e-01,4
9,RecSinClima_espaciotemporal_7d,Rec sin clima (43-46),espaciotemporal,7,kWh/dia,MM30,Media móvil de 30 días,64.44,False,3233.924614,...,0.197,0.251,0.429,0.058,1.000000e+00,NO_SIGNIFICATIVO,Rec sin clima (43-46)__MM30,4.029670e-01,4.029670e-01,3


=== Model Confidence Set (alpha=0.10): ASHR sobrevive en 27/28 escenarios ===


,escenario,suite,task,n_time_units,n_survivors,ashr_in_mcs_90,p_mcs_ashr,survivors
0,D1_espaciotemporal_cinco_variables,D+1 (37-40),espaciotemporal,262,1,True,1.00,ASHR
1,D1_espaciotemporal_sin_meteorologia,D+1 (37-40),espaciotemporal,262,9,True,1.00,"BayesianRidge, ELAC, IWLAR, KLAC, Lasso, RARC,..."
2,D1_intraplanta_cinco_variables,D+1 (37-40),intraplanta,262,8,False,0.04,"HARC, IWLAR, MARB, PDEC, RARC, RLAC, RQRF, WCFC"
3,D1_intraplanta_sin_meteorologia,D+1 (37-40),intraplanta,262,14,True,1.00,"BayesianRidge, ELAC, HARC, KLAC, Lasso, MARB, ..."
4,RecSinClima_espaciotemporal_7d,Rec sin clima (43-46),espaciotemporal,126,15,True,1.00,"ABGPR, BayesianRidge, ELAC, KLAC, Lasso, MARB,..."
5,RecSinClima_intraplanta_7d,Rec sin clima (43-46),intraplanta,134,9,True,1.00,"BayesianRidge, Lasso, Media móvil de 30 días, ..."
6,RecSinClima_espaciotemporal_14d,Rec sin clima (43-46),espaciotemporal,63,15,True,1.00,"ABGPR, BayesianRidge, ELAC, KLAC, Lasso, MARB,..."
7,RecSinClima_intraplanta_14d,Rec sin clima (43-46),intraplanta,67,22,True,1.00,"ABGPR, BayesianRidge, DecisionTree, ELAC, HARC..."
8,RecSinClima_espaciotemporal_21d,Rec sin clima (43-46),espaciotemporal,41,14,True,1.00,"BayesianRidge, ELAC, KLAC, Lasso, MARB, Media ..."
9,RecSinClima_intraplanta_21d,Rec sin clima (43-46),intraplanta,44,22,True,1.00,"ABGPR, BayesianRidge, DecisionTree, ELAC, HARC..."


=== Prueba global SPA (post-hoc, B=500) ===


,escenario,suite,task,n_time_units,p_spa_H0_MM30,p_spa_H0_ASHR
0,D1_espaciotemporal_cinco_variables,D+1 (37-40),espaciotemporal,262,0.000,1.000
1,D1_espaciotemporal_sin_meteorologia,D+1 (37-40),espaciotemporal,262,0.000,1.000
2,D1_intraplanta_cinco_variables,D+1 (37-40),intraplanta,262,0.000,0.000
3,D1_intraplanta_sin_meteorologia,D+1 (37-40),intraplanta,262,0.000,1.000
4,RecSinClima_espaciotemporal_7d,Rec sin clima (43-46),espaciotemporal,126,1.000,0.154
5,RecSinClima_intraplanta_7d,Rec sin clima (43-46),intraplanta,134,0.706,0.546
6,RecSinClima_espaciotemporal_14d,Rec sin clima (43-46),espaciotemporal,63,1.000,0.124
7,RecSinClima_intraplanta_14d,Rec sin clima (43-46),intraplanta,67,0.702,0.662
8,RecSinClima_espaciotemporal_21d,Rec sin clima (43-46),espaciotemporal,41,1.000,0.138
9,RecSinClima_intraplanta_21d,Rec sin clima (43-46),intraplanta,44,1.000,0.538


## 11. Diseño efectivo, dependencia temporal y análisis de errores por planta

Se evalúa $T_{\text{eff}} \ge 2H$ (excluido de inferencia confirmatoria en `13/28` escenarios principalmente de horizontes `21d` y `28d`) y se resume la distribución de errores por planta (`17` centrales entre `9.8` y `388.0 MW`).

In [9]:
if bundle is not None:
    design_df = analyze_effective_design(bundle)
    display(design_df)
    plant_summary = (
        bundle["tabla_por_planta"]
        .groupby("code", as_index=False)
        .agg(
            n_escenarios=("escenario", "count"),
            mae_ashr_medio=("MAE_ASHR", "mean"),
            mae_mm30_medio=("MAE_MM30", "mean"),
            escenarios_mejor_que_mm30=("diff_vs_MM30", lambda s: int((s < 0).sum())),
        )
        .sort_values("mae_ashr_medio", ascending=False)
    )
    display(plant_summary)
else:
    design_df = pd.DataFrame()
    print("Diagnóstico de diseño efectivo T_eff: PENDIENTE DE ESTADO B.")

,escenario,suite,task,horizon_days,T_eff,t_eff_insufficient,nota_potencia
0,D1_espaciotemporal_cinco_variables,D+1 (37-40),espaciotemporal,1,152.56,False,T_eff >= 2*H; considerar dependencia espacial ...
1,D1_espaciotemporal_sin_meteorologia,D+1 (37-40),espaciotemporal,1,180.06,False,T_eff >= 2*H; considerar dependencia espacial ...
2,D1_intraplanta_cinco_variables,D+1 (37-40),intraplanta,1,116.59,False,T_eff >= 2*H; considerar dependencia espacial ...
3,D1_intraplanta_sin_meteorologia,D+1 (37-40),intraplanta,1,195.27,False,T_eff >= 2*H; considerar dependencia espacial ...
4,RecSinClima_espaciotemporal_7d,Rec sin clima (43-46),espaciotemporal,7,71.54,False,T_eff >= 2*H; considerar dependencia espacial ...
5,RecSinClima_intraplanta_7d,Rec sin clima (43-46),intraplanta,7,134.00,False,T_eff >= 2*H; considerar dependencia espacial ...
6,RecSinClima_espaciotemporal_14d,Rec sin clima (43-46),espaciotemporal,14,40.87,False,T_eff >= 2*H; considerar dependencia espacial ...
7,RecSinClima_intraplanta_14d,Rec sin clima (43-46),intraplanta,14,37.80,False,T_eff >= 2*H; considerar dependencia espacial ...
8,RecSinClima_espaciotemporal_21d,Rec sin clima (43-46),espaciotemporal,21,41.00,True,T_eff < 2*H: aproximación asintótica poco fiable
9,RecSinClima_intraplanta_21d,Rec sin clima (43-46),intraplanta,21,22.93,True,T_eff < 2*H: aproximación asintótica poco fiable


,code,n_escenarios,mae_ashr_medio,mae_mm30_medio,escenarios_mejor_que_mm30
14,GYPO,28,1.309544e+06,1.268199e+06,15
0,3DDT,28,5.527398e+05,4.496038e+05,14
7,3IZ6,28,3.517922e+05,3.569092e+05,18
15,MATA,28,3.178618e+05,2.919436e+05,24
6,3IRX,28,3.153462e+05,2.883520e+05,8
3,3HF5,28,2.976040e+05,2.917151e+05,21
16,TPUY,28,2.925292e+05,2.887312e+05,11
13,EPFV,28,2.849253e+05,2.226463e+05,12
4,3INX,28,1.784369e+05,1.509590e+05,20
5,3IQA,28,8.202571e+04,8.045925e+04,15


## 12. Evaluador formal de superioridad (R3)

Evalúa por separado las cuatro condiciones exigidas (`CUMPLE`, `NO_CUMPLE`, `NO_VERIFICABLE`, `PENDIENTE_AUTORIZACION`), manteniendo un veredicto estrictamente conservador.

In [10]:
if bundle is not None:
    r3_verdict_df = evaluate_r3_superiority(bundle, scope_authorized=SCOPE_AUTHORIZED, alpha=0.05)
    display(r3_verdict_df)
else:
    r3_verdict_df = pd.DataFrame()
    print("Evaluador formal R3: PENDIENTE DE ESTADO B.")

,condicion,estado,evidencia,calculo,fuente,limitacion,resultado
0,Condicion_1_Metrica_Externa,NO_CUMPLE,Victorias vs Mejor Biblioteca: 18/28 escenarios,pct_vs_MejorLib < 0,tabla_28_escenarios,Requiere victoria consistente sin ocultar derr...,18/28
1,Condicion_2_Pruebas_Multiples_y_MCS,NO_VERIFICABLE,Solo 2 procedimientos ejecutados (DM-HLN Holm=...,DM-HLN-HAC + Holm + MCS (alpha=0.10) + SPA (GW...,"tabla_inferencia, tabla_mcs, tabla_spa",Exige 3 procedimientos distintos calibrados; G...,"Holm=2, MCS_excl=1"
2,Condicion_3_Tamano_Efecto,NO_CUMPLE,|Cohen's d| >= 0.2 en 0/15 escenarios con T_ef...,|d| >= 0.20 (y umbral Cliff |delta| >= 0.147),tabla_inferencia,Efectos pequeños pueden carecer de relevancia ...,0/15
3,Condicion_4_Alcance_y_Consistencia,PENDIENTE_AUTORIZACION,28 escenarios solares (17 plantas Colombia); s...,Validación de alcance docente + consistencia m...,config.SCENARIOS_28,Sustitución de datasets externos pendiente de ...,PENDIENTE_AUTORIZACION


## 13. Estudios diagnósticos complementarios: ablación controlada (`D+1` real)

Estudio diagnóstico ejecutado sobre el panel real `D+1` con partición cronológica estricta (`max_train_date < min_val_date`, sin fuga temporal). No sustituye el OOF confirmatorio de 28 escenarios; aísla la contribución marginal de la normalización a factor de capacidad (`CF`), la pérdida de Huber, la regularización estructurada y las restricciones de monotonía.

In [11]:
abl_path = OOF_DIR / "ablaciones" / "tabla_ablaciones.csv"
abl_df = pd.DataFrame()
if bundle is not None and abl_path.exists():
    abl_df = pd.read_csv(abl_path)
    display(abl_df)
    plot_ablation_bar(abl_df, FIG_DIR)
else:
    print("Tabla de ablaciones: PENDIENTE (ejecutar `python -m sm_cfhr.cli report`).")

,ablation_variant,mae_cf,mae_kwh
0,SM-CFHR_completo,0.034878,60631.214754
1,sin_huber_L2,0.034961,60784.002789
2,sin_l2_estructurada,0.034870,60606.282762
3,sin_monotonia,0.034878,60631.214754
4,escala_std_vs_mad,0.034878,60631.214754
5,energia_bruta_vs_cf,0.230646,270035.482533


## 14. Sensibilidad de hiperparámetros, semillas, estrés ante atípicos y escalabilidad (`D+1` real)

Incluye barridos 1D de `delta`, `alpha_lin` y `alpha_meteo`, estabilidad en 10 semillas (`2026..2035`), contaminación sintética sobre validación limpia (`0%`, `5%`, `10%` de atípicos) y curvas de aprendizaje/tiempo de ajuste.

In [12]:
sens_path = OOF_DIR / "sensibilidad" / "tabla_sensibilidad_hp.csv"
seed_path = OOF_DIR / "sensibilidad" / "tabla_estabilidad_semillas.csv"
stress_path = OOF_DIR / "robustez" / "tabla_estres_atipicos.csv"
lc_path = OOF_DIR / "robustez" / "tabla_aprendizaje_escalabilidad.csv"
if bundle is not None and sens_path.exists() and stress_path.exists() and lc_path.exists():
    sens_df = pd.read_csv(sens_path)
    stress_df = pd.read_csv(stress_path)
    lc_df = pd.read_csv(lc_path)
    display(sens_df)
    if seed_path.exists():
        seed_df = pd.read_csv(seed_path)
        display(seed_df.groupby("variant", as_index=False)["mae_macro_kwh"].agg(["mean", "std", "min", "max"]))
    display(stress_df)
    display(lc_df)
    plot_hp_sensitivity(sens_df, FIG_DIR)
    plot_robustness_stress(stress_df, FIG_DIR)
    plot_learning_and_scalability(lc_df, FIG_DIR)
else:
    print("Estudios de sensibilidad y robustez: PENDIENTE (ejecutar `python -m sm_cfhr.cli report`).")

,varied_param,param_value,mae,rmse
0,delta,0.5000,0.034812,0.045993
1,delta,1.0000,0.034838,0.045836
2,delta,1.3500,0.034878,0.045787
3,delta,2.0000,0.034932,0.045769
4,delta,3.0000,0.034960,0.045785
5,alpha_lin,0.0001,0.034985,0.045913
6,alpha_lin,0.0010,0.034965,0.045891
7,alpha_lin,0.0100,0.034878,0.045787
8,alpha_lin,0.1000,0.034906,0.045667
9,alpha_lin,1.0000,0.036391,0.046754


,variant,mean,std,min,max
0,SM-CFHR-Fixed,56437.847767,0.000000,56437.847767,56437.847767
1,SM-CFHR-HPO,57300.572869,1223.093377,55660.910683,59124.801099


,model,outlier_frac,mae,rmse
0,SM-CFHR_Huber,0.00,0.034878,0.045787
1,Referencia_L2,0.00,0.034961,0.045785
2,SM-CFHR_Huber,0.05,0.034880,0.045771
3,Referencia_L2,0.05,0.035544,0.046995
4,SM-CFHR_Huber,0.10,0.034912,0.045833
5,Referencia_L2,0.10,0.037487,0.050176


,fraction,n_train,n_features,mae,fit_time_s,predict_time_s,peak_memory_kb
0,0.10,704,17,0.279701,0.011944,0.000153,265.125977
1,0.25,1759,17,0.044135,0.011241,0.000140,561.829102
2,0.50,3518,17,0.034795,0.012335,0.000149,1056.547852
3,1.00,7036,17,0.034878,0.013476,0.000142,2045.985352


## 15. Incertidumbre predictiva: intervalos conformes temporales (`D+1` real)

Cuantiles conformes de muestra finita calibrados estrictamente con residuos pasados (`val_cal` anterior en fecha a `val_test`) para coberturas nominales **80 %** y **90 %**, reportando cobertura empírica real y subcobertura ante cambio de régimen temporal.

In [13]:
conf_path = OOF_DIR / "incertidumbre" / "tabla_conformal_resumen.csv"
conf_plant_path = OOF_DIR / "incertidumbre" / "tabla_conformal_por_planta.csv"
if bundle is not None and conf_path.exists():
    conf_df = pd.read_csv(conf_path)
    display(conf_df)
    if conf_plant_path.exists():
        display(pd.read_csv(conf_plant_path))
    plot_conformal_coverage(conf_df, FIG_DIR)
else:
    print("Calibración conformal temporal: PENDIENTE (ejecutar `python -m sm_cfhr.cli report`).")

,nominal_coverage,alpha,conformal_halfwidth,mean_width,n_cal,empirical_coverage
0,0.9,0.1,0.066183,0.132366,1720,0.832412
1,0.8,0.2,0.047970,0.095941,1720,0.692478


,nominal_coverage,plant,n_obs,empirical_coverage,mean_width
0,0.9,3DDT,105,0.819048,0.132366
1,0.9,3EDL,107,0.663551,0.132366
2,0.9,3EFY,107,0.654206,0.132366
3,0.9,3HF5,104,0.865385,0.132366
4,0.9,3INX,106,0.943396,0.132366
5,0.9,3IQA,107,0.962617,0.132366
6,0.9,3IRX,107,0.878505,0.132366
7,0.9,3IZ6,107,0.859813,0.132366
8,0.9,3J2B,107,0.775701,0.132366
9,0.9,3K8C,105,0.971429,0.132366


## 16. Interpretabilidad global, restricciones monótonas y descomposición aditiva local

En `SM-CFHR` (modelo paramétrico lineal en predictores estandarizados), la descomposición $\phi_{q,j} = \hat{\sigma}_c (z_{q,j} - \bar{z}_{\mathrm{train},j})\hat{w}_j$ es **exacta** por instancia y verifica directamente las restricciones monótonas $\hat{w}_{\mathrm{cf30}} \ge 0$ y $\hat{w}_{\mathrm{rad\_pron}} \ge 0$.

In [14]:
interp_path = OOF_DIR / "interpretabilidad" / "tabla_importancia_global.csv"
local_path = OOF_DIR / "interpretabilidad" / "ejemplos_locales.json"
if bundle is not None and interp_path.exists():
    interp_df = pd.read_csv(interp_path)
    mono_ok = bool((interp_df.loc[interp_df["feature"].isin(["cf30", "rad_pron"]), "std_coef"] >= 0.0).all())
    print("Restricciones monótonas (cf30 >= 0 y rad_pron >= 0) verificadas:", mono_ok)
    display(interp_df)
    if local_path.exists():
        display(pd.DataFrame(json.loads(local_path.read_text(encoding="utf-8"))).drop(columns=["attributions"], errors="ignore"))
    plot_feature_importance(interp_df, FIG_DIR)
else:
    print("Interpretabilidad global/local: PENDIENTE (ejecutar `python -m sm_cfhr.cli report`).")

Restricciones monótonas (cf30 >= 0 y rad_pron >= 0) verificadas: True


,feature,std_coef,mean_abs_contrib
0,cf30,0.459142,0.025500
1,anom_rad,0.173094,0.010687
2,anom_nube,-0.172986,0.010673
3,month_cos,-0.110761,0.007015
4,doy_sin,-0.090567,0.005283
5,doy_cos,0.078546,0.005277
6,rad_pron,0.086332,0.005004
7,anom_humedad,-0.075225,0.004657
8,month_sin,0.068461,0.004213
9,anom_temp,0.067225,0.004183


,row_idx,baseline_pred,model_pred,sum_attributions
0,0,0.247022,0.318969,0.071947
1,1763,0.247022,0.211387,-0.035635
2,3527,0.247022,0.341567,0.094545


## 17. Coste computacional medido y escalabilidad paralela

Se distinguen explícitamente:
- **Tiempo de pared real:** `SM-CFHR-Fixed` (`499.748 s` = `8.33 min`, `8` workers, `241` reciclajes, RSS agregado pico `2,067.62 MB`); `SM-CFHR-HPO` (`29,550.293 s` = `8.21 h` acumuladas en dos sesiones con reanudación verificada tras traslado, `12` workers, `228` reciclajes, RSS agregado pico `4,201.10 MB`).
- **Suma de tiempos acumulados por escenario (`tabla_coste`):** suma del tiempo de cómputo de los workers en paralelo (supera al tiempo de pared).
- **Benchmark separado de escalabilidad SMT (`48` unidades de desarrollo, `25` trials):** `8w` (`70.064 s`, `41.11 u/min`), `10w` (`65.841 s`, `+6.41%`), `12w` (`63.360 s`, `+10.58%`, `3,465.09 MB` RSS agregado, `53.1 °C`). *Fixed (8w) y HPO (12w) no constituyen una comparación controlada a igualdad de paralelismo.*

In [15]:
if bundle is not None:
    m_fixed_p = ROOT / "results" / "oof_v1_0_0" / "oof_v1_0_3_fixed" / "consolidated" / "manifest_ashr_oof.json"
    m_hpo_p = ROOT / "results" / "oof_v1_0_0" / "oof_v1_0_3_hpo" / "consolidated" / "manifest_ashr_oof.json"
    cost_summary = pd.DataFrame([
        {"variante": "SM-CFHR-Fixed (B=0)", "workers": 8, "unidades": 14625, "tiempo_pared_s": 499.748, "tiempo_pared_min": round(499.748 / 60.0, 2), "reciclajes": 241, "rss_agregado_pico_mb": 2067.62},
        {"variante": "SM-CFHR-HPO (B=25, 2 sesiones)", "workers": 12, "unidades": 14625, "tiempo_pared_s": 29550.293, "tiempo_pared_min": round(29550.293 / 60.0, 2), "reciclajes": 228, "rss_agregado_pico_mb": 4201.10},
    ])
    display(cost_summary)
    if m_hpo_p.exists():
        m_hpo = json.loads(m_hpo_p.read_text(encoding="utf-8"))
        t_hpo = m_hpo.get("tiempos_por_escenario_s", {})
        c_df = bundle["tabla_coste"].copy().rename(columns={"tiempo_s": "tiempo_acum_workers_fixed_s"})
        c_df["tiempo_acum_workers_hpo_s"] = c_df["scenario_id"].map(lambda s: float(t_hpo.get(f"hpo__{s}", np.nan)))
        display(c_df)

,variante,workers,unidades,tiempo_pared_s,tiempo_pared_min,reciclajes,rss_agregado_pico_mb
0,SM-CFHR-Fixed (B=0),8,14625,499.748,8.33,241,2067.62
1,"SM-CFHR-HPO (B=25, 2 sesiones)",12,14625,29550.293,492.50,228,4201.10


,scenario_id,suite,task,tiempo_acum_workers_fixed_s,tiempo_acum_workers_hpo_s
0,D1_espaciotemporal_cinco_variables,D+1 (37-40),espaciotemporal,11.478,135.185
1,D1_espaciotemporal_sin_meteorologia,D+1 (37-40),espaciotemporal,6.796,103.593
2,D1_intraplanta_cinco_variables,D+1 (37-40),intraplanta,7.967,263.831
3,D1_intraplanta_sin_meteorologia,D+1 (37-40),intraplanta,7.664,209.579
4,RecSinClima_espaciotemporal_7d,Rec sin clima (43-46),espaciotemporal,38.075,17483.858
5,RecSinClima_intraplanta_7d,Rec sin clima (43-46),intraplanta,69.642,33944.541
6,RecSinClima_espaciotemporal_14d,Rec sin clima (43-46),espaciotemporal,25.766,9975.312
7,RecSinClima_intraplanta_14d,Rec sin clima (43-46),intraplanta,40.303,18654.436
8,RecSinClima_espaciotemporal_21d,Rec sin clima (43-46),espaciotemporal,19.914,7117.162
9,RecSinClima_intraplanta_21d,Rec sin clima (43-46),intraplanta,30.065,13785.848


## 18. Exportación integral automática y resumen descriptivo

In [16]:
resumen_desc = build_descriptive_summary(bundle)
print(json.dumps(resumen_desc, indent=2, ensure_ascii=False))
if bundle is not None:
    exported_paper = export_paper_tables(bundle, TAB_DIR)
    full_manifest = export_full_scientific_artifacts(
        bundle,
        OOF_DIR,
        extra_artifacts={
            "conclusiones/veredicto_r3": r3_verdict_df,
            "inferencia/diseno_efectivo": design_df,
            "conclusiones/resumen_descriptivo": resumen_desc,
        },
    )
    print("Exportación completa:", full_manifest["n_archivos_exportados"], "archivos con SHA-256.")

{
  "estado": "VALIDO",
  "procedencia": "OOF_DEFINITIVO_REAL",
  "n_escenarios": 28,
  "victorias_vs_mejor_bib": 18,
  "derrotas_vs_mejor_bib": 10,
  "victorias_vs_mejor_custom": 22,
  "victorias_vs_mm30": 15,
  "rango_medio_en_23": 2.93,
  "mejor_escenario_vs_bib": "D1_espaciotemporal_cinco_variables (-22.16%)",
  "peor_escenario_vs_bib": "DirectoSinClima_intraplanta_28d (+24.24%)",
  "sig_holm_vs_bib": 2,
  "t_eff_insuficiente_n": 13
}


Exportación completa: 25 archivos con SHA-256.


## 19. Conclusiones científicas verificadas, limitaciones y reproducibilidad

1. **Desempeño empírico real (`SM-CFHR-Fixed`, $B=0$):** obtiene rango medio **`2.93 / 23`**, **`18 / 28`** victorias descriptivas frente al mejor modelo de biblioteca (`22 / 28` frente al mejor modelo custom, `15 / 28` frente a `MM30`) y permanece en el MCS ($lpha=0{,}10$) en **`27 / 28`** escenarios. Destaca especialmente en **generalización espaciotemporal (`ET`)** con meteorología (`D1 ET 5 vars`: `-22.16%` vs mejor biblioteca; `RecGEFSv12 ET 7–28d`: rango medio `1.0` y `-11.84%` medio), mientras que pierde sistemáticamente en **`DirectoSinClima` intraplanta** (`+13.27%` medio, hasta `+24.24%` en `28d`).
2. **`SM-CFHR-Fixed` vs. `SM-CFHR-HPO` ($B=25$):** sobre el soporte común alineado, `HPO` mejora a `Fixed` únicamente en **`9 / 28`** escenarios (todos `intraplanta`), mientras `Fixed` gana en `19 / 28` (`14 / 14` en `ET`) con mejor rango medio (`2.93` vs `3.86`) y un coste computacional **`59×` menor en tiempo de pared** (`8.33 min` vs `8.21 h`). Esto respalda el uso de `SM-CFHR-Fixed` como configuración principal parsimoniosa.
3. **Inferencia estadística y veredicto R3:** solo **`2 / 15`** escenarios con $T_{\text{eff}} \ge 2H$ resultan significativos tras Holm–Bonferroni frente al mejor modelo de biblioteca y `0 / 15` superan $|d_{\mathrm{Cohen}}| \ge 0{,}20$. En consecuencia, **no se declara superioridad estadística general** (`R3`: `Condicion_1: NO_CUMPLE`, `Condicion_2: NO_VERIFICABLE`, `Condicion_3: NO_CUMPLE`, `Condicion_4: PENDIENTE_AUTORIZACION`).
4. **Estudios diagnósticos complementarios (`D+1` real):** la normalización adimensional por capacidad instalada (`CF`) es el componente estructural más determinante (`0.0349` vs `0.2306` CF al usar energía bruta); la pérdida de Huber preserva estabilidad ante `5%–10%` de contaminación (`0.03488 -> 0.03491` CF vs `0.03496 -> 0.03749` en `L2`); las restricciones monótonas (`cf30 >= 0`, `rad_pron >= 0`) se cumplen con `cf30` y `anom_rad` como principales predictores; y los intervalos conformes temporales muestran subcobertura moderada (`83.2%` para `90%` nominal; `69.2%` para `80%` nominal) atribuible a deriva temporal entre bloques.
5. **Reproducibilidad sin reentrenar:** ejecutar `python -m sm_cfhr.cli report` y luego `python scripts/evaluacion/generar_notebook_ashr.py` + `jupyter nbconvert --to notebook --execute --inplace notebooks/actuales/evaluacion_cientifica_ashr_oof.ipynb`.